# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Display high-level metadata
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print("\nKeywords:")
pprint.pprint(metadata.keywords)


## 2. Data Overview
Review available record sets, their IDs (`@id`), fields (`@id`), and columns (`@id`).

The `mlcroissant` library allows us to inspect the structure of the dataset. All entities are referenced by their `@id` (unique identifier).

In [ ]:
# List all record sets and their fields, using @id
print("Available record sets (with @id):\n--------------------------")
record_sets = list(dataset.record_sets())  # Returns RecordSetMetadata objects
for rs in record_sets:
    print(f"- Record Set Name: {rs.name} (id: {rs.id}) — description: {rs.description[:70] if rs.description else ''}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    • {field.name} (id: {field.id}, type: {field.data_type})")
        # List columns if available
        if hasattr(field, 'columns') and field.columns:
            for column in field.columns:
                print(f"      - Column: {column.name} (id: {column.id}, type: {column.data_type})")
    print()


## 3. Data Extraction
Load records from specific record sets into DataFrames using their `@id`.

> **Pro tip:** Refer to the record set, field, and column `@id`s from the overview above when extracting data.

In [ ]:
# Choose record sets by @id (update these if needed after reviewing the previous output)
# For FAIR², common record set names are 'Protein Table', 'Protein Occurrences', etc.

# Let's list out the available record set ids
record_set_ids = [rs.id for rs in record_sets]
print('Record set IDs:', record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    # Load all records for the given record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for {record_set_id}")

# Show columns for first record set
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_record_set_id:
    print(f"\nColumns in main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply basic data analysis operations using relevant fields.

- We'll filter, normalize, and group based on numeric/categorical columns in the main record set.

In [ ]:
# Pick a numeric field for EDA (update field id to match your dataset)
df = dataframes[main_record_set_id]
# Example: try 'MW' (molecular weight) or 'Coverage_percentage' as numeric field, else infer
numeric_candidates = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(df[col])]
print("Numeric field candidates:", numeric_candidates)

if len(numeric_candidates) == 0:
    print("No numeric columns found for EDA.")
else:
    numeric_field = numeric_candidates[0]  # Pick the first numerically-typed field
    print(f"Using numeric field for demonstration: {numeric_field}\n")
    threshold = df[numeric_field].quantile(0.75)  # Use upper quartile as threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Choose a group field if available
    # Example: group by "Description" or similar
    group_field_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        print(f"\nGrouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
        display(grouped_df.head())
    else:
        print("No suitable categorical field for grouping found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_candidates) > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    if group_field_candidates:
        plt.figure(figsize=(10,5))
        top_categories = df[group_field].value_counts().index[:8]
        sns.boxplot(data=df[df[group_field].isin(top_categories)], x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} distribution by {group_field} (top 8 categories)')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load a complex Croissant dataset using `mlcroissant`
- Explore the schema using entity `@id`s for record sets, fields, and columns
- Extract data dynamically and perform exploratory analysis (filtering, normalization, grouping)
- Visualize distributions and category relationships

You can now extend this workflow for deeper domain-specific analyses, reproducible reporting, or machine learning tasks with the FAIR² dataset!